<a href="https://colab.research.google.com/github/Hschimera79/peptide-nasal-delivery-screen/blob/main/03_descriptors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Read the uploaded variant table
import pandas as pd
df = pd.read_csv("variants.csv")
print(df.shape)
df.head()


(98, 4)


,id,parent,change,risk
0,Gln4Ala,desmopressin,Gln4 -> Ala,NaN
1,Gln4Gly,desmopressin,Gln4 -> Gly,NaN
2,Gln4Ser,desmopressin,Gln4 -> Ser,NaN
3,Gln4Thr,desmopressin,Gln4 -> Thr,NaN
4,Gln4Val,desmopressin,Gln4 -> Val,NaN


In [2]:
# Install the cheminformatics and peptide descriptor libraries
!pip install rdkit peptides --quiet

In [3]:
# Rebuild each variant's one-letter sequence from its mutation label
THREE_TO_ONE = {
    "Ala": "A", "Gly": "G", "Ser": "S", "Thr": "T", "Val": "V",
    "Leu": "L", "Ile": "I", "Pro": "P", "Phe": "F", "Tyr": "Y",
    "Trp": "W", "Asn": "N", "Gln": "Q", "Asp": "D", "Glu": "E",
    "Lys": "K", "Arg": "R", "His": "H", "Met": "M", "Cys": "C",
    "Mpa": "C",
}

BASE = ["Mpa", "Tyr", "Phe", "Gln", "Asn", "Cys", "Pro", "Arg", "Gly"]

def sequence_from_change(change):
    residues = list(BASE)
    if change.startswith("L-"):
        return "".join(THREE_TO_ONE[r] for r in residues)
    old_part, new_aa = change.split(" -> ")
    pos = int("".join(ch for ch in old_part if ch.isdigit()))
    residues[pos - 1] = new_aa
    return "".join(THREE_TO_ONE[r] for r in residues)

df["seq"] = df["change"].apply(sequence_from_change)
df[["id", "change", "seq"]].head()

,id,change,seq
0,Gln4Ala,Gln4 -> Ala,CYFANCPRG
1,Gln4Gly,Gln4 -> Gly,CYFGNCPRG
2,Gln4Ser,Gln4 -> Ser,CYFSNCPRG
3,Gln4Thr,Gln4 -> Thr,CYFTNCPRG
4,Gln4Val,Gln4 -> Val,CYFVNCPRG


In [4]:
# Compute sequence-based physicochemical descriptors for every variant
import peptides

def describe(seq):
    p = peptides.Peptide(seq)
    return {
        "mw": p.molecular_weight(),
        "charge_ph6": p.charge(pH=6.0),
        "isoelectric_point": p.isoelectric_point(),
        "hydrophobicity": p.hydrophobicity(scale="KyteDoolittle"),
        "hydrophobic_moment": p.hydrophobic_moment(),
        "instability_index": p.instability_index(),
    }

desc = df["seq"].apply(describe).apply(pd.Series)
df = pd.concat([df, desc], axis=1)
df.head()

,id,parent,change,risk,seq,mw,charge_ph6,isoelectric_point,hydrophobicity,hydrophobic_moment,instability_index
0,Gln4Ala,desmopressin,Gln4 -> Ala,NaN,CYFANCPRG,1030.18414,0.986802,8.222829,-0.188889,0.416030,9.277778
1,Gln4Gly,desmopressin,Gln4 -> Gly,NaN,CYFGNCPRG,1016.15724,0.986802,8.222829,-0.433333,0.424823,-0.155556
2,Gln4Ser,desmopressin,Gln4 -> Ser,NaN,CYFSNCPRG,1046.18354,0.986802,8.222829,-0.477778,0.471007,9.277778
3,Gln4Thr,desmopressin,Gln4 -> Thr,NaN,CYFTNCPRG,1060.21044,0.986802,8.222829,-0.466667,0.461354,-7.422222
4,Gln4Val,desmopressin,Gln4 -> Val,NaN,CYFVNCPRG,1058.23794,0.986802,8.222829,0.077778,0.390122,9.277778


In [5]:
# Save the enriched table for the filtering module
df.to_csv("descriptors.csv", index=False)
print(df.shape)
df.describe()

(98, 11)


,mw,charge_ph6,isoelectric_point,hydrophobicity,hydrophobic_moment,instability_index
count,98.000000,98.000000,98.000000,98.000000,98.000000,98.000000
mean,1095.774724,0.809008,7.621277,-0.527211,0.451534,21.529592
std,45.327328,0.616809,1.158485,0.366960,0.126342,21.627959
min,988.100440,-1.008751,3.749955,-1.233333,0.090579,-22.533333
25%,1061.458890,0.124906,6.413053,-0.777778,0.415112,9.277778
50%,1092.241290,0.986802,8.222829,-0.533333,0.513943,18.711111
75%,1127.290740,0.986802,8.222829,-0.241667,0.527471,32.463889
max,1216.397340,1.986802,8.803861,0.222222,0.675640,97.522222


In [6]:
# Drop duplicated columns caused by running the concat cell more than once
df = df.loc[:, ~df.columns.duplicated()]
print(df.shape)
print(df.columns.tolist())


(98, 11)
['id', 'parent', 'change', 'risk', 'seq', 'mw', 'charge_ph6', 'isoelectric_point', 'hydrophobicity', 'hydrophobic_moment', 'instability_index']
